Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import copy
from torchsummary import summary

Get Dataset Paths

In [2]:
train_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\train"
val_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\val"
test_dir = r"C:\SUTD\50.021 Artificial Intelligence\pppppp\Master_Dataset\test"

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    print('Using CPU')

Using GPU: NVIDIA GeForce RTX 4060 Laptop GPU


Transforms

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


DataLoaders

In [5]:
# Load datasets
train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

print("Classes:", train_dataset.classes)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

Classes: ['NORMAL', 'PNEUMONIA']
Train size: 14054
Val size: 1757
Test size: 1757


In [6]:
# Sanity check one sample
sample, label = train_dataset[0]
print("Sample shape:", sample.shape)
print("Label index:", label)
print("Class name:", train_dataset.classes[label])

Sample shape: torch.Size([3, 224, 224])
Label index: 0
Class name: NORMAL


Evaluation Function for all Models

In [7]:
def evaluate_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)

            # In eval mode, GoogLeNet usually returns main logits only
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)

    return epoch_loss, acc, f1, auc

GOOGLENET MODEL

In [8]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchsummary import summary

googlenetmodel = models.googlenet(pretrained=True)

for param in googlenetmodel.parameters():
    param.requires_grad = False

googlenetmodel.fc = nn.Sequential(
    nn.Linear(googlenetmodel.fc.in_features, 512),
    nn.Dropout(p=0.3),
    nn.ReLU(inplace=True),
    nn.Linear(512, 128),
    nn.ReLU(inplace=True),
    nn.Linear(128, 32),
    nn.ReLU(inplace=True),
    nn.Linear(32, 2)
)

for name, param in googlenetmodel.named_parameters():
    if "inception5" in name:
        param.requires_grad = True

for param in googlenetmodel.fc.parameters():
    param.requires_grad = True


# Chuyển mô hình sang thiết bị (CPU hoặc GPU)
googlenetmodel = googlenetmodel.to(device)

# In ra cấu trúc mô hình
print(summary(googlenetmodel, (3, 224, 224)))

c:\Users\thoai\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\thoai\AppData\Local\Programs\Python\Python311\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=GoogLeNet_Weights.IMAGENET1K_V1`. You can also use `weights=GoogLeNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
       BasicConv2d-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]           4,096
       BatchNorm2d-6           [-1, 64, 56, 56]             128
       BasicConv2d-7           [-1, 64, 56, 56]               0
            Conv2d-8          [-1, 192, 56, 56]         110,592
       BatchNorm2d-9          [-1, 192, 56, 56]             384
      BasicConv2d-10          [-1, 192, 56, 56]               0
        MaxPool2d-11          [-1, 192, 28, 28]               0
           Conv2d-12           [-1, 64, 28, 28]          12,288
      BatchNorm2d-13           [-1, 64, 28, 28]             128
      BasicConv2d-14           [-1, 64,

Googlenet Training parameters

In [9]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(googlenetmodel.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=4)

Training Loop for Googlenet

In [10]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

best_val_loss = 100.0
best_model_params = None

num_epochs = 15

for epoch in range(num_epochs):
    googlenetmodel.train()
    running_loss = 0.0
    total_train = 0
    correct_train = 0

    with tqdm(total=len(train_loader), desc=f'Epoch [{epoch+1}/{num_epochs}]') as pbar:
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = googlenetmodel(inputs)

            # ✅ Handle GoogLeNet outputs
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

            pbar.update(1)
            pbar.set_postfix({'Loss': running_loss / total_train})

    epoch_loss = running_loss / len(train_dataset)
    train_acc = correct_train / total_train

    # ================= VALIDATION =================
    googlenetmodel.eval()

    val_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = googlenetmodel(inputs)

            # ✅ Handle GoogLeNet outputs
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_loss = val_loss / len(val_dataset)

    # ✅ Metrics
    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds)
    val_auc = roc_auc_score(all_labels, all_probs)

    print(
        f'Epoch [{epoch+1}/{num_epochs}] | '
        f'Train Loss: {epoch_loss:.4f} | '
        f'Train Acc: {train_acc:.4f} | '
        f'Val Loss: {val_loss:.4f} | '
        f'Val Acc: {val_acc:.4f} | '
        f'Val F1: {val_f1:.4f} | '
        f'Val AUC: {val_auc:.4f}'
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_params = googlenetmodel.state_dict()

    scheduler.step(val_loss)


# ================= LOAD BEST MODEL =================
if best_model_params is not None:
    googlenetmodel.load_state_dict(best_model_params)

Epoch [1/15]:   0%|          | 0/440 [00:00<?, ?it/s]

Epoch [1/15]: 100%|██████████| 440/440 [00:25<00:00, 17.33it/s, Loss=0.511]


Epoch [1/15] | Train Loss: 0.5110 | Train Acc: 0.7493 | Val Loss: 0.4458 | Val Acc: 0.7888 | Val F1: 0.7938 | Val AUC: 0.8711


Epoch [2/15]: 100%|██████████| 440/440 [00:17<00:00, 25.36it/s, Loss=0.424]


Epoch [2/15] | Train Loss: 0.4244 | Train Acc: 0.8013 | Val Loss: 0.4430 | Val Acc: 0.7866 | Val F1: 0.7965 | Val AUC: 0.8732


...

Epoch [13/15] | Train Loss: 0.0136 | Train Acc: 0.9954 | Val Loss: 1.0753 | Val Acc: 0.7729 | Val F1: 0.7713 | Val AUC: 0.8574


Epoch [14/15]: 100%|██████████| 440/440 [00:17<00:00, 24.76it/s, Loss=0.0121] 


Epoch [14/15] | Train Loss: 0.0121 | Train Acc: 0.9962 | Val Loss: 1.0772 | Val Acc: 0.7723 | Val F1: 0.7732 | Val AUC: 0.8570


Epoch [15/15]: 100%|██████████| 440/440 [00:17<00:00, 25.19it/s, Loss=0.0111] 


Epoch [15/15] | Train Loss: 0.0111 | Train Acc: 0.9964 | Val Loss: 1.1023 | Val Acc: 0.7706 | Val F1: 0.7760 | Val AUC: 0.8573


Save Model 

In [11]:
torch.save(googlenetmodel.state_dict(), "googlenet_finetuned_baseline.pth")
print("Model saved.")

Model saved.


ALEXNET MODEL

In [12]:
alexnetmodel = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)

alexnetmodel.classifier = nn.Sequential(
    nn.Dropout(),
    nn.Linear(9216, 4096),  
    nn.ReLU(inplace=True),
    nn.Dropout(),
    nn.Linear(4096, 1024),  
    nn.ReLU(inplace=True),
    nn.Linear(1024, 512), 
    nn.ReLU(inplace=True),
    nn.Linear(512, 128), 
    nn.ReLU(inplace=True),
    nn.Linear(128, 32),
    nn.ReLU(inplace=True),
    nn.Linear(32, 2)
)

for param in alexnetmodel.parameters():
    param.requires_grad = False

for param in alexnetmodel.classifier.parameters():
    param.requires_grad = True

alexnetmodel.to(device)
print(summary(alexnetmodel, (3, 224, 224)))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 64, 55, 55]          23,296
              ReLU-2           [-1, 64, 55, 55]               0
         MaxPool2d-3           [-1, 64, 27, 27]               0
            Conv2d-4          [-1, 192, 27, 27]         307,392
              ReLU-5          [-1, 192, 27, 27]               0
         MaxPool2d-6          [-1, 192, 13, 13]               0
            Conv2d-7          [-1, 384, 13, 13]         663,936
              ReLU-8          [-1, 384, 13, 13]               0
            Conv2d-9          [-1, 256, 13, 13]         884,992
             ReLU-10          [-1, 256, 13, 13]               0
           Conv2d-11          [-1, 256, 13, 13]         590,080
             ReLU-12          [-1, 256, 13, 13]               0
        MaxPool2d-13            [-1, 256, 6, 6]               0
AdaptiveAvgPool2d-14            [-1, 25

Alexnet parameters

In [13]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(alexnetmodel.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=4)

Training loop for Alexnet

In [14]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

best_val_loss = 100.0
best_model_params = None

num_epochs = 15

for epoch in range(num_epochs):
    alexnetmodel.train()
    running_loss = 0.0
    total_train = 0
    correct_train = 0

    with tqdm(total=len(train_loader), desc=f'Epoch [{epoch+1}/{num_epochs}]') as pbar:
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = alexnetmodel(inputs)

            # ✅ Handle GoogLeNet outputs
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

            pbar.update(1)
            pbar.set_postfix({'Loss': running_loss / total_train})

    epoch_loss = running_loss / len(train_dataset)
    train_acc = correct_train / total_train

    # ================= VALIDATION =================
    alexnetmodel.eval()

    val_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = alexnetmodel(inputs)

            # ✅ Handle GoogLeNet outputs
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_loss = val_loss / len(val_dataset)

    # ✅ Metrics
    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds)
    val_auc = roc_auc_score(all_labels, all_probs)

    print(
        f'Epoch [{epoch+1}/{num_epochs}] | '
        f'Train Loss: {epoch_loss:.4f} | '
        f'Train Acc: {train_acc:.4f} | '
        f'Val Loss: {val_loss:.4f} | '
        f'Val Acc: {val_acc:.4f} | '
        f'Val F1: {val_f1:.4f} | '
        f'Val AUC: {val_auc:.4f}'
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_params = alexnetmodel.state_dict()

    scheduler.step(val_loss)


# ================= LOAD BEST MODEL =================
if best_model_params is not None:
    alexnetmodel.load_state_dict(best_model_params)

Epoch [1/15]: 100%|██████████| 440/440 [00:16<00:00, 26.67it/s, Loss=0.503]


Epoch [1/15] | Train Loss: 0.5027 | Train Acc: 0.7539 | Val Loss: 0.4370 | Val Acc: 0.7849 | Val F1: 0.7881 | Val AUC: 0.8767


Epoch [2/15]: 100%|██████████| 440/440 [00:16<00:00, 27.00it/s, Loss=0.454]


Epoch [2/15] | Train Loss: 0.4541 | Train Acc: 0.7811 | Val Loss: 0.4419 | Val Acc: 0.7900 | Val F1: 0.7810 | Val AUC: 0.8814


Epoch [3/15]: 100%|██████████| 440/440 [00:15<00:00, 27.98it/s, Loss=0.441]


...

Epoch [13/15] | Train Loss: 0.3618 | Train Acc: 0.8267 | Val Loss: 0.4215 | Val Acc: 0.8008 | Val F1: 0.8049 | Val AUC: 0.8904


Epoch [14/15]: 100%|██████████| 440/440 [00:15<00:00, 28.59it/s, Loss=0.354]


Epoch [14/15] | Train Loss: 0.3537 | Train Acc: 0.8330 | Val Loss: 0.4293 | Val Acc: 0.7917 | Val F1: 0.8047 | Val AUC: 0.8908


Epoch [15/15]: 100%|██████████| 440/440 [00:15<00:00, 28.07it/s, Loss=0.344]


Epoch [15/15] | Train Loss: 0.3436 | Train Acc: 0.8401 | Val Loss: 0.4352 | Val Acc: 0.7997 | Val F1: 0.8022 | Val AUC: 0.8869


Save model

In [15]:
torch.save(alexnetmodel.state_dict(), "alexnet_finetuned_baseline.pth")
print("Model saved.")

Model saved.


RESNET-18 MODEL

In [16]:
resnet18model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

resnet18model.fc = nn.Sequential(
    nn.Dropout(),
    nn.Linear(512, 128),
    nn.ReLU(inplace=True),
    nn.Linear(128, 32),
    nn.ReLU(inplace=True),
    nn.Linear(32, 2),
    nn.ReLU(inplace=True)
)

# freeze everything
for param in resnet18model.parameters():
    param.requires_grad = False

# unfreeze final layer
for param in resnet18model.fc.parameters():
    param.requires_grad = True

resnet18model.to(device)
print(summary(resnet18model, (3, 224, 224)))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 64, 112, 112]           9,408
       BatchNorm2d-2         [-1, 64, 112, 112]             128
              ReLU-3         [-1, 64, 112, 112]               0
         MaxPool2d-4           [-1, 64, 56, 56]               0
            Conv2d-5           [-1, 64, 56, 56]          36,864
       BatchNorm2d-6           [-1, 64, 56, 56]             128
              ReLU-7           [-1, 64, 56, 56]               0
            Conv2d-8           [-1, 64, 56, 56]          36,864
       BatchNorm2d-9           [-1, 64, 56, 56]             128
             ReLU-10           [-1, 64, 56, 56]               0
       BasicBlock-11           [-1, 64, 56, 56]               0
           Conv2d-12           [-1, 64, 56, 56]          36,864
      BatchNorm2d-13           [-1, 64, 56, 56]             128
             ReLU-14           [-1, 64,

Resnet 18 parameters

In [17]:
from torch.optim.lr_scheduler import ReduceLROnPlateau
criterion = nn.CrossEntropyLoss()
# Định nghĩa optimizer và scheduler
optimizer = optim.Adam(resnet18model.parameters(), lr=0.0001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=4)

Training loop for resnet-18

In [18]:
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

best_val_loss = 100.0
best_model_params = None

num_epochs = 15

for epoch in range(num_epochs):
    resnet18model.train()
    running_loss = 0.0
    total_train = 0
    correct_train = 0

    with tqdm(total=len(train_loader), desc=f'Epoch [{epoch+1}/{num_epochs}]') as pbar:
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = resnet18model(inputs)

            # ✅ Handle GoogLeNet outputs
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

            pbar.update(1)
            pbar.set_postfix({'Loss': running_loss / total_train})

    epoch_loss = running_loss / len(train_dataset)
    train_acc = correct_train / total_train

    # ================= VALIDATION =================
    resnet18model.eval()

    val_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = resnet18model(inputs)

            # ✅ Handle GoogLeNet outputs
            if hasattr(outputs, "logits"):
                outputs = outputs.logits
            elif isinstance(outputs, tuple):
                outputs = outputs[0]

            loss = criterion(outputs, labels)
            val_loss += loss.item() * inputs.size(0)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_loss = val_loss / len(val_dataset)

    # ✅ Metrics
    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds)
    val_auc = roc_auc_score(all_labels, all_probs)

    print(
        f'Epoch [{epoch+1}/{num_epochs}] | '
        f'Train Loss: {epoch_loss:.4f} | '
        f'Train Acc: {train_acc:.4f} | '
        f'Val Loss: {val_loss:.4f} | '
        f'Val Acc: {val_acc:.4f} | '
        f'Val F1: {val_f1:.4f} | '
        f'Val AUC: {val_auc:.4f}'
    )

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_params = resnet18model.state_dict()

    scheduler.step(val_loss)


# ================= LOAD BEST MODEL =================
if best_model_params is not None:
    resnet18model.load_state_dict(best_model_params)


Epoch [1/15]: 100%|██████████| 440/440 [00:14<00:00, 30.62it/s, Loss=0.624]


Epoch [1/15] | Train Loss: 0.6244 | Train Acc: 0.6624 | Val Loss: 0.5564 | Val Acc: 0.7205 | Val F1: 0.7037 | Val AUC: 0.7959


Epoch [2/15]: 100%|██████████| 440/440 [00:14<00:00, 31.08it/s, Loss=0.566]


Epoch [2/15] | Train Loss: 0.5664 | Train Acc: 0.7172 | Val Loss: 0.5316 | Val Acc: 0.7456 | Val F1: 0.7438 | Val AUC: 0.8142


Epoch [3/15]: 100%|██████████| 440/440 [00:17<00:00, 25.44it/s, Loss=0.561]


...

Epoch [13/15] | Train Loss: 0.5231 | Train Acc: 0.7341 | Val Loss: 0.4981 | Val Acc: 0.7592 | Val F1: 0.7531 | Val AUC: 0.8403


Epoch [14/15]: 100%|██████████| 440/440 [00:14<00:00, 31.04it/s, Loss=0.524]


Epoch [14/15] | Train Loss: 0.5240 | Train Acc: 0.7354 | Val Loss: 0.4912 | Val Acc: 0.7604 | Val F1: 0.7743 | Val AUC: 0.8408


Epoch [15/15]: 100%|██████████| 440/440 [00:14<00:00, 31.26it/s, Loss=0.523]


Epoch [15/15] | Train Loss: 0.5229 | Train Acc: 0.7357 | Val Loss: 0.4873 | Val Acc: 0.7644 | Val F1: 0.7677 | Val AUC: 0.8433


In [19]:
torch.save(resnet18model.state_dict(), "resnet18_finetuned_baseline.pth")
print("Model saved.")

Model saved.


Save model